In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os

DATA_PATH = 'C:/Git projects/ObjectRegconition/data/raw/ecg.csv'
MODEL_PATH = 'C:/Git projects/ObjectRegconition/autoencoder_model.pth'
OUTPUT_PATH = 'C:/Git projects/ObjectRegconition/predictions_results.csv'

class AnomalyDetector(nn.Module):
    def __init__(self, input_dim=140):
        super(AnomalyDetector, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def load_and_preprocess_data(path):
    dataframe = pd.read_csv(path, header=None)
    raw_data = dataframe.values
    labels = raw_data[:, -1]
    data = raw_data[:, 0:-1]
    
    train_data, test_data, train_labels, test_labels = train_test_split(
        data, labels, test_size=0.2, random_state=21
    )
    
    min_val = np.min(train_data)
    max_val = np.max(train_data)
    
    train_data = (train_data - min_val) / (max_val - min_val)
    test_data = (test_data - min_val) / (max_val - min_val)
    
    return train_data.astype(np.float32), test_data.astype(np.float32), train_labels, test_labels

def train_and_save_model(train_data, epochs=2000, batch_size=512, learning_rate=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    model = AnomalyDetector(input_dim=train_data.shape[1]).to(device)
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    train_tensor = torch.tensor(train_data)
    dataset = TensorDataset(train_tensor, train_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    print(f"Training autoencoder on {len(train_data)} normal samples...")
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for batch_data, batch_target in dataloader:
            batch_data = batch_data.to(device)
            batch_target = batch_target.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_data)
            loss = criterion(outputs, batch_target)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.6f}")
    
    torch.save({
        'model_state_dict': model.state_dict(),
        'input_dim': train_data.shape[1]
    }, MODEL_PATH)
    print(f"Model saved to {MODEL_PATH}")
    
    return model, device

def load_model():
    checkpoint = torch.load(MODEL_PATH)
    input_dim = checkpoint['input_dim']
    model = AnomalyDetector(input_dim=input_dim)
    model.load_state_dict(checkpoint['model_state_dict'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    return model, device

def predict(model, test_data, device):
    model.eval()
    test_tensor = torch.tensor(test_data).to(device)
    
    with torch.no_grad():
        reconstructions = model(test_tensor).cpu().numpy()
    
    reconstruction_error = np.mean(np.abs(test_data - reconstructions), axis=1)
    
    return reconstructions, reconstruction_error

def main():
    print("="*60)
    print("ECG Anomaly Detection - Autoencoder (PyTorch)")
    print("="*60)
    
    print("\nLoading data...")
    train_data, test_data, train_labels, test_labels = load_and_preprocess_data(DATA_PATH)
    
    is_normal_train = train_labels == 1.0
    is_normal_test = test_labels == 1.0
    
    train_normal = train_data[is_normal_train]
    
    print(f"Train samples: {len(train_data)} (Normal: {len(train_normal)})")
    print(f"Test samples: {len(test_data)}")
    
    if not os.path.exists(MODEL_PATH):
        print("\nTraining model...")
        model, device = train_and_save_model(train_normal, epochs=50)
    else:
        print("\nLoading trained model...")
        model, device = load_model()
    
    print("\nMaking predictions...")
    reconstructions, errors = predict(model, test_data, device)
    
    normal_errors = errors[is_normal_test]
    abnormal_errors = errors[~is_normal_test]
    
    threshold = np.mean(normal_errors) + 2 * np.std(normal_errors)
    
    predicted_is_abnormal = errors > threshold
    
    true_is_abnormal = ~is_normal_test
    
    correct = (predicted_is_abnormal == true_is_abnormal).astype(int)
    
    results_df = pd.DataFrame({
        'sample_index': range(len(test_data)),
        'true_label': ['Abnormal' if x else 'Normal' for x in true_is_abnormal],
        'predicted_label': ['Abnormal' if x else 'Normal' for x in predicted_is_abnormal],
        'reconstruction_error': errors,
        'threshold': threshold,
        'correct': correct
    })
    
    results_df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nResults saved to {OUTPUT_PATH}")
    
    print("\n" + "="*60)
    print("PREDICTIONS vs TRUE RESULTS")
    print("="*60)
    
    print(f"\n{'Sample':<10} {'True':<12} {'Predicted':<12} {'Error':<12} {'Match'}")
    print("-"*55)
    for i in range(min(20, len(results_df))):
        row = results_df.iloc[i]
        print(f" {i:<8} {row['true_label']:<12} {row['predicted_label']:<12} {row['reconstruction_error']:.6f}   {'Yes' if row['correct'] else 'No'}")
    
    print(f"\n... showing first 20 of {len(results_df)} samples")
    
    acc = accuracy_score(true_is_abnormal, predicted_is_abnormal)
    prec = precision_score(true_is_abnormal, predicted_is_abnormal)
    rec = recall_score(true_is_abnormal, predicted_is_abnormal)
    f1 = f1_score(true_is_abnormal, predicted_is_abnormal)
    
    cm = confusion_matrix(true_is_abnormal, predicted_is_abnormal)
    
    print("\n" + "="*60)
    print("METRICS")
    print("="*60)
    
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")
    
    print(f"\nConfusion Matrix:")
    print(f"                      Predicted")
    print(f"                 Normal    Abnormal")
    print(f"Actual Normal    {cm[0,0]:5d}    {cm[0,1]:5d}")
    print(f"       Abnormal {cm[1,0]:5d}    {cm[1,1]:5d}")
    
    print("\n" + "="*60)
    print("ERROR COMPARISON")
    print("="*60)
    print(f"\nNormal (true):   Mean={np.mean(normal_errors):.6f}, Std={np.std(normal_errors):.6f}")
    print(f"Abnormal (true): Mean={np.mean(abnormal_errors):.6f}, Std={np.std(abnormal_errors):.6f}")
    print(f"\nThreshold: {threshold:.6f}")

if __name__ == "__main__":
    main()

ECG Anomaly Detection - Autoencoder (PyTorch)

Loading data...
Train samples: 3998 (Normal: 2359)
Test samples: 1000

Loading trained model...

Making predictions...

Results saved to C:/Git projects/ObjectRegconition/predictions_results.csv

PREDICTIONS vs TRUE RESULTS

Sample     True         Predicted    Error        Match
-------------------------------------------------------
 0        Normal       Normal       0.028290   Yes
 1        Abnormal     Abnormal     0.042198   Yes
 2        Normal       Normal       0.008303   Yes
 3        Normal       Normal       0.022303   Yes
 4        Abnormal     Abnormal     0.052984   Yes
 5        Normal       Abnormal     0.056574   No
 6        Normal       Normal       0.012924   Yes
 7        Normal       Normal       0.009632   Yes
 8        Abnormal     Abnormal     0.042169   Yes
 9        Normal       Normal       0.011190   Yes
 10       Normal       Abnormal     0.063483   No
 11       Normal       Normal       0.010491   Yes
 12   

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

RESULTS_PATH = 'C:/Git projects/ObjectRegconition/predictions_results.csv'

def visualize_results():
    df = pd.read_csv(RESULTS_PATH)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('ECG Autoencoder - Prediction Results Visualization', fontsize=14, fontweight='bold')
    
    ax1 = axes[0, 0]
    normal_errors = df[df['true_label'] == 'Normal']['reconstruction_error']
    abnormal_errors = df[df['true_label'] == 'Abnormal']['reconstruction_error']
    
    ax1.hist(normal_errors, bins=40, alpha=0.7, label='Normal (True)', color='green', edgecolor='darkgreen')
    ax1.hist(abnormal_errors, bins=40, alpha=0.7, label='Abnormal (True)', color='red', edgecolor='darkred')
    threshold = df['threshold'].iloc[0]
    ax1.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={threshold:.4f}')
    ax1.set_xlabel('Reconstruction Error', fontsize=11)
    ax1.set_ylabel('Frequency', fontsize=11)
    ax1.set_title('Distribution of Reconstruction Errors', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[0, 1]
    cm = confusion_matrix(df['true_label'], df['predicted_label'], labels=['Normal', 'Abnormal'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Abnormal'])
    disp.plot(ax=ax2, cmap='Blues', values_format='d')
    ax2.set_title('Confusion Matrix', fontsize=12)
    
    ax3 = axes[1, 0]
    correct_normal = ((df['true_label'] == 'Normal') & (df['predicted_label'] == 'Normal')).sum()
    correct_abnormal = ((df['true_label'] == 'Abnormal') & (df['predicted_label'] == 'Abnormal')).sum()
    wrong_normal_wrong = ((df['true_label'] == 'Normal') & (df['predicted_label'] == 'Abnormal')).sum()
    wrong_abnormal_wrong = ((df['true_label'] == 'Abnormal') & (df['predicted_label'] == 'Normal')).sum()
    
    categories = ['Correct\nNormal', 'Correct\nAbnormal', 'Wrong\n(Normal→Abn)', 'Wrong\n(Abn→Normal)']
    values = [correct_normal, correct_abnormal, wrong_normal_wrong, wrong_abnormal_wrong]
    colors_bar = ['green', 'orange', 'red', 'darkred']
    
    bars = ax3.bar(categories, values, color=colors_bar, edgecolor='black', alpha=0.8)
    ax3.set_ylabel('Count', fontsize=11)
    ax3.set_title('Prediction Results Breakdown', fontsize=12)
    
    for bar, val in zip(bars, values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, str(val), 
                ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax4 = axes[1, 1]
    samples = range(50)
    threshold = df['threshold'].iloc[0]
    
    for i, (_, row) in enumerate(df.head(50).iterrows()):
        color = 'green' if row['true_label'] == 'Normal' else 'red'
        ax4.bar(i, row['reconstruction_error'], color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    
    ax4.axhline(threshold, color='blue', linestyle='--', linewidth=2, label=f'Threshold')
    ax4.set_xlabel('Sample Index', fontsize=11)
    ax4.set_ylabel('Reconstruction Error', fontsize=11)
    ax4.set_title('Error by Sample (First 50)\nGreen=Actual Normal, Red=Actual Abnormal', fontsize=12)
    ax4.legend()
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('C:/Git projects/ObjectRegconition/predictions_visualization.png', dpi=150, bbox_inches='tight')
    print(f"Visualization saved to predictions_visualization.png")
    plt.close()

def main():
    df = pd.read_csv(RESULTS_PATH)
    
    total = len(df)
    correct_normal = ((df['true_label'] == 'Normal') & (df['predicted_label'] == 'Normal')).sum()
    correct_abnormal = ((df['true_label'] == 'Abnormal') & (df['predicted_label'] == 'Abnormal')).sum()
    wrong_normal = ((df['true_label'] == 'Normal') & (df['predicted_label'] == 'Abnormal')).sum()
    wrong_abnormal = ((df['true_label'] == 'Abnormal') & (df['predicted_label'] == 'Normal')).sum()
    
    accuracy = (correct_normal + correct_abnormal) / total * 100
    
    print("="*60)
    print("PREDICTION RESULTS SUMMARY")
    print("="*60)
    print(f"\nTotal samples: {total}")
    print(f"Correct predictions: {correct_normal + correct_abnormal}")
    print(f"Incorrect predictions: {wrong_normal + wrong_abnormal}")
    print(f"Accuracy: {accuracy:.2f}%")
    
    print("\n" + "-"*50)
    print("CONFUSION MATRIX")
    print("-"*50)
    print(f"\n                 Predicted")
    print(f"                 Normal    Abnormal")
    print(f"Actual Normal    {correct_normal:5d}    {wrong_normal:5d}")
    print(f"       Abnormal {wrong_abnormal:5d}    {correct_abnormal:5d}")
    
    print("\n" + "-"*50)
    print("ERROR STATISTICS")
    print("-"*50)
    normal_errors = df[df['true_label'] == 'Normal']['reconstruction_error']
    abnormal_errors = df[df['true_label'] == 'Abnormal']['reconstruction_error']
    
    print(f"Normal samples:   Mean={normal_errors.mean():.6f}, Std={normal_errors.std():.6f}")
    print(f"Abnormal samples: Mean={abnormal_errors.mean():.6f}, Std={abnormal_errors.std():.6f}")
    print(f"\nThreshold used: {df['threshold'].iloc[0]:.6f}")
    
    visualize_results()

if __name__ == "__main__":
    main()

PREDICTION RESULTS SUMMARY

Total samples: 1000
Correct predictions: 955
Incorrect predictions: 45
Accuracy: 95.50%

--------------------------------------------------
CONFUSION MATRIX
--------------------------------------------------

                 Predicted
                 Normal    Abnormal
Actual Normal      536       24
       Abnormal    21      419

--------------------------------------------------
ERROR STATISTICS
--------------------------------------------------
Normal samples:   Mean=0.017524, Std=0.010465
Abnormal samples: Mean=0.048477, Std=0.007483

Threshold used: 0.038435
Visualization saved to predictions_visualization.png


In [3]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

DATA_PATH = 'C:/Git projects/ObjectRegconition/data/raw/ecg.csv'
MODEL_PATH = 'C:/Git projects/ObjectRegconition/autoencoder_model.pth'

class AnomalyDetector(nn.Module):
    def __init__(self, input_dim=140):
        super(AnomalyDetector, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

def load_data():
    dataframe = pd.read_csv(DATA_PATH, header=None)
    raw_data = dataframe.values
    labels = raw_data[:, -1]
    data = raw_data[:, 0:-1]
    
    _, test_data, _, test_labels = train_test_split(
        data, labels, test_size=0.2, random_state=21
    )
    
    min_val = np.min(test_data) if len(test_data) > 0 else 0
    max_val = np.max(test_data) if len(test_data) > 0 else 1
    test_data = (test_data - min_val) / (max_val - min_val)
    
    return test_data.astype(np.float32), test_labels

def load_model():
    checkpoint = torch.load(MODEL_PATH)
    input_dim = checkpoint['input_dim']
    model = AnomalyDetector(input_dim=input_dim)
    model.load_state_dict(checkpoint['model_state_dict'])
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    model.eval()
    return model, device

def visualize_ecg_comparison(test_data, test_labels):
    model, device = load_model()
    
    test_tensor = torch.tensor(test_data).to(device)
    
    with torch.no_grad():
        reconstructions = model(test_tensor).cpu().numpy()
    
    test_labels = test_labels == 1.0
    
    normal_indices = np.where(test_labels)[0][:3]
    abnormal_indices = np.where(~test_labels)[0][:3]
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    fig.suptitle('ECG Signal Comparison: Original vs Reconstructed (Autoencoder)', fontsize=14, fontweight='bold')
    
    colors_original = ['green', 'blue', 'purple']
    colors_recon = ['lightgreen', 'lightskyblue', 'violet']
    
    for i, idx in enumerate(normal_indices):
        ax = axes[i, 0]
        original = test_data[idx]
        reconstructed = reconstructions[idx]
        
        ax.plot(original, color='green', alpha=0.7, linewidth=1.5, label='Original')
        ax.plot(reconstructed, color='red', alpha=0.7, linewidth=1.5, linestyle='--', label='Reconstructed')
        ax.set_title(f'Normal Sample #{idx} (Error: {np.mean(np.abs(original - reconstructed)):.6f})', fontsize=10)
        ax.set_xlabel('Time Step')
        ax.set_ylabel('Amplitude')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    for i, idx in enumerate(abnormal_indices):
        ax = axes[i, 1]
        original = test_data[idx]
        reconstructed = reconstructions[idx]
        
        ax.plot(original, color='blue', alpha=0.7, linewidth=1.5, label='Original')
        ax.plot(reconstructed, color='red', alpha=0.7, linewidth=1.5, linestyle='--', label='Reconstructed')
        ax.set_title(f'Abnormal Sample #{idx} (Error: {np.mean(np.abs(original - reconstructed)):.6f})', fontsize=10)
        ax.set_xlabel('Time Step')
        ax.set_ylabel('Amplitude')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('C:/Git projects/ObjectRegconition/ecg_comparison.png', dpi=150, bbox_inches='tight')
    print(f"ECG comparison saved to ecg_comparison.png")
    plt.close()

def visualize_reconstruction_errors(test_data, test_labels):
    model, device = load_model()
    
    test_tensor = torch.tensor(test_data).to(device)
    
    with torch.no_grad():
        reconstructions = model(test_tensor).cpu().numpy()
    
    errors = np.mean(np.abs(test_data - reconstructions), axis=1)
    test_labels = test_labels == 1.0
    
    threshold = np.mean(errors[test_labels]) + 2 * np.std(errors[test_labels])
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Reconstruction Error Analysis', fontsize=14, fontweight='bold')
    
    ax1 = axes[0, 0]
    normal_errors = errors[test_labels]
    abnormal_errors = errors[~test_labels]
    
    ax1.hist(normal_errors, bins=30, alpha=0.7, label='Normal', color='green', edgecolor='darkgreen')
    ax1.hist(abnormal_errors, bins=30, alpha=0.7, label='Abnormal', color='red', edgecolor='darkred')
    ax1.axvline(threshold, color='black', linestyle='--', linewidth=2, label=f'Threshold={threshold:.4f}')
    ax1.set_xlabel('Reconstruction Error')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Error Distribution')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2 = axes[0, 1]
    indices = range(100)
    colors = ['green' if l else 'red' for l in test_labels[:100]]
    ax2.bar(indices, errors[:100], color=colors, alpha=0.7)
    ax2.axhline(threshold, color='black', linestyle='--', linewidth=2, label='Threshold')
    ax2.set_xlabel('Sample Index')
    ax2.set_ylabel('Reconstruction Error')
    ax2.set_title('Error per Sample (First 100)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    ax3 = axes[1, 0]
    box_data = [normal_errors, abnormal_errors]
    bp = ax3.boxplot(box_data, tick_labels=['Normal', 'Abnormal'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightgreen')
    bp['boxes'][1].set_facecolor('lightcoral')
    ax3.axhline(threshold, color='black', linestyle='--', linewidth=2, label='Threshold')
    ax3.set_ylabel('Reconstruction Error')
    ax3.set_title('Error Box Plot')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    ax4 = axes[1, 1]
    pred_abnormal = errors > threshold
    true_abnormal = ~test_labels
    
    correct_normal = np.sum((~pred_abnormal) & (~true_abnormal))
    wrong_normal = np.sum(pred_abnormal & (~true_abnormal))
    correct_abnormal = np.sum(pred_abnormal & true_abnormal)
    wrong_abnormal = np.sum((~pred_abnormal) & true_abnormal)
    
    categories = ['Correct\nNormal', 'Correct\nAbnormal', 'Wrong\n(Norm→Abn)', 'Wrong\n(Abn→Norm)']
    values = [correct_normal, correct_abnormal, wrong_normal, wrong_abnormal]
    colors_bar = ['green', 'orange', 'red', 'darkred']
    
    bars = ax4.bar(categories, values, color=colors_bar, edgecolor='black', alpha=0.8)
    ax4.set_ylabel('Count')
    ax4.set_title('Prediction Breakdown')
    
    for bar, val in zip(bars, values):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(val), 
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('C:/Git projects/ObjectRegconition/error_analysis.png', dpi=150, bbox_inches='tight')
    print(f"Error analysis saved to error_analysis.png")
    plt.close()

def main():
    print("Loading ECG data and model...")
    test_data, test_labels = load_data()
    print(f"Test samples: {len(test_data)}")
    
    print("\nGenerating ECG comparison plots...")
    visualize_ecg_comparison(test_data, test_labels)
    
    print("\nGenerating error analysis plots...")
    visualize_reconstruction_errors(test_data, test_labels)
    
    print("\nDone!")

if __name__ == "__main__":
    main()

Loading ECG data and model...
Test samples: 1000

Generating ECG comparison plots...
ECG comparison saved to ecg_comparison.png

Generating error analysis plots...
Error analysis saved to error_analysis.png

Done!
